# 第70章 交互面积图（px.area）

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 7 / 18 步：交互观察分布与矩阵**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 交互气泡图  →  **本章任务：** 交互面积图（px.area）  →  **下一步：** 交互直方图（px.histogram）
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

面积图最擅长展现一条数据随时间累积的轨迹，当我们同时想知道总量有多大、中间是怎么波动的时候，折线图只给了边界，柱状图只有离散的点，而面积图用填充的色块让趋势一眼可感。在日常报表里，从电商销售额的月度走势到网站在线人数的变化，面积图都是盯着一条曲线看它走向哪里最直观的选择。特别值得推荐的是 Plotly 的交互版本：鼠标划到某个时点，就能立刻读出那个位置的精确数值，对初学分析的同学来说，更容易把图形直觉和数据事实对上号。


## 本章目标

学完本章，你将能够：

- **理解**：理解「交互面积图（px.area）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「交互面积图（px.area）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「交互面积图（px.area）」并读出其中的结论。


## 适用场景

**背景引入**：面积图最擅长展现一条数据随时间累积的轨迹，当我们同时想知道总量有多大、中间是怎么波动的时候，折线图只给了边界，柱状图只有离散的点，而面积图用填充的色块让趋势一眼可感。在日常报表里，从电商销售额的月度走势到网站在线人数的变化，面积图都是盯着一条曲线看它走向哪里最直观的选择。特别值得推荐的是 Plotly 的交互版本：鼠标划到某个时点，就能立刻读出那个位置的精确数值，对初学分析的同学来说，更容易把图形直觉和数据事实对上号。（可以把它想成往杯子里倒水：折线是杯子的轮廓，面积是里面水的多少；多类叠起来就是几层水，最上层水面是总量，每层厚度是各部分占多少。）

强调随时间变化的总量、区间或组成。


## 数据结构

有序X、非负Y和可选分类列。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 groupnorm="fraction" 改为 groupnorm=None，对比百分比构成与绝对量堆积
2. 修改 hovermode="x unified" 为 "closest"，观察悬浮信息聚合方式的变化
3. 添加 hovertemplate 自定义悬停信息格式，说明交互提示对某时点精确值读取的作用


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `px.area()`、`fig.update_layout()`、`fig.show()` | 强调随时间变化的总量、区间或组成。 | 中间层难以比较 |
| 进阶变体 | `pd.DataFrame()`、`np.repeat()`、`px.area()`、`fig.update_layout()` | 在基础图表上增加分组、注释、布局或交互 | 序列有负值 |
| 关键参数 | `groupnorm` | 百分比归一化 | 中间层难以比较 |
| 关键参数 | `line_group` | 序列 | 序列有负值 |
| 关键参数 | `color` | 堆积层 | 图例顺序与堆积顺序不一致 |
| 关键参数 | `hovermode` | 悬浮 | 中间层难以比较 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-70 -->
### 数学推导｜面积堆叠必须满足组成恒等式

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜同一时点先统一粒度。** 各组数值为 $x_{1,t},\ldots,x_{G,t}$。

**第 2 步｜逐层累加形成边界。** 第 $g$ 层上边界为

$$
H_{g,t}=\sum_{j=1}^{g}x_{j,t}
$$

**第 3 步｜最上层必须回到总体。** $H_{G,t}=T_t$；若画百分比堆叠，则每层厚度为 $s_{g,t}=x_{g,t}/T_t$ 且总和为 1。

**把上面的关系收束为本章计算式：**

$$
T_t=\sum_{g=1}^{G}x_{g,t},\qquad s_{g,t}=\frac{x_{g,t}}{T_t}
$$

**符号解释：** $x_{g,t}$ 是组 $g$ 在时点 $t$ 的值，$T_t$ 是同一时点总体。

**代码对应：** 绘图前按时间透视为宽表，并检查各层之和是否等于总体。

**使用边界：** 堆叠顺序影响可读性；除最底层外，其他类别不适合比较细小变化。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(
    f'Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行'
)


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.area(
    monthly, x="month", y="sales", markers=True, title="上半年销售额面积图"
)
fig.update_layout(
    xaxis_title="月份", yaxis_title="销售额（万元）", hovermode="x unified"
)
fig.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：上面这张面积图把 sales 映射到了 y 轴。试着只修改一个字段或一个展示参数，观察图形发生的变化：把 y 换成 monthly 的 profit（3 期滚动平均），或者把 markers 从 True 改成 False，再或者改一下标题。下面留好了脚手架，把你想替换的字段名填进 y_col、把标记开关填进 markers_flag，先自己跑一遍体会区别，再点开答案对照自检。


In [ ]:
try:
    # 请在下方填写代码：练一练，把 y 换成 profit，或把 markers 改为 False，再观察变化。
    import plotly.express as px
    import pandas as pd

    y_col = "sales"  # TODO: 可改成 "profit"
    markers_flag = True  # TODO: 可改成 False
    fig = px.area(
        monthly,
        x="month",
        y=y_col,
        markers=markers_flag,
        title="面积图：字段与展示参数对比",
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
composition = pd.DataFrame(
    {
        "month": np.repeat(monthly["month"], 3),
        "category": ["办公", "数码", "家居"] * len(monthly),
        "sales": [
            38,
            52,
            30,
            42,
            65,
            41,
            40,
            60,
            39,
            48,
            78,
            50,
            55,
            92,
            58,
            59,
            105,
            64,
            38,
            52,
            30,
            42,
            65,
            41,
            40,
            60,
            39,
            48,
            78,
            50,
            55,
            92,
            58,
            59,
            105,
            64,
        ],
    }
)
fig = px.area(
    composition, x="month", y="sales", color="category", title="品类销售构成"
)
fig.update_layout(
    xaxis_title="月份",
    yaxis_title="销售额（万元）",
    legend_title="品类",
    hovermode="x unified",
)
fig.show()


## 参数说明

- groupnorm：百分比归一化
- line_group：序列
- color：堆积层
- hovermode：悬浮


## 结果解读

顶部边界表示总量，各层厚度表示组成；Hover可查看某时点的精确值。


## 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
_demo_fig = px.bar(report, x="region", y="sales", title="地区销售额")
_demo_fig.show()


### 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
_demo_fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
_demo_fig.update_traces(textposition="outside")
_demo_fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
_demo_fig.show()


### 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 中间层难以比较
- 序列有负值
- 图例顺序与堆积顺序不一致


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：按月份叠加多条面积，比较不同口径的构成
    # 【目标】用 melt 把多列转成长表，用 color 叠加多条面积，练习看构成。
    import plotly.express as px

    # 起点示例(已可运行)：把销售额与利润画成两条面积，观察构成变化。
    monthly_long = monthly.melt(
        id_vars="month",
        value_vars=["sales", "profit"],
        var_name="指标",
        value_name="金额",
    )
    fig = px.area(
        monthly_long,
        x="month",
        y="金额",
        color="指标",
        title="销售额与利润面积对比",
    )
    fig.update_layout(
        xaxis_title="月份", yaxis_title="金额（万元）", hovermode="x unified"
    )
    fig.show()

    # ---- 反思记录：两条面积叠加，哪个序列更容易比较 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

用交互面积图展示连续趋势的累计量或多类别构成。


### 你已经掌握

- 判断交互面积图（px.area）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `groupnorm` | 百分比归一化 |
| `line_group` | 序列 |
| `color` | 堆积层 |
| `hovermode` | 悬浮 |


### 需要注意

- 中间层难以比较
- 序列有负值
- 图例顺序与堆积顺序不一致


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
import pandas as pd
import plotly.express as px

# 完整答案：把 y 换成 profit（3 期滚动平均），关闭 markers，改标题突出对比
y_col = "profit"
markers_flag = False
fig = px.area(
    monthly,
    x="month",
    y=y_col,
    markers=markers_flag,
    title="销售额 3 期滚动平均——面积图",
)
fig.update_layout(
    xaxis_title="月份", yaxis_title="平均值", hovermode="x unified"
)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
share_data = regional.copy()
fig = px.area(
    share_data,
    x="region",
    y="sales",
    color="channel",
    groupnorm="fraction",
    title="区域渠道百分比构成",
)
fig.update_layout(
    xaxis_title="区域", yaxis_title="构成比例", legend_title="渠道"
)
fig.show()
